# Sakha GRPO Training Notebook

Train a hospital ward assistant using GRPO (Group Relative Policy Optimization) on the Sakha environment.

**Requirements**: GPU runtime (T4 is sufficient for 0.6B model)

**What this does**:
1. Installs Sakha environment and dependencies
2. Configures GRPO training with HF TRL
3. Runs training for N episodes
4. Plots reward curves and metrics

## 1. Setup - Install Dependencies

In [ ]:
# Clone the Sakha repository (colab-training branch for testing)
!git clone -b colab-training https://github.com/atharva-again/sakha.git
%cd sakha

In [ ]:
# Install uv and dependencies
!pip install uv -q
!uv pip install --system -e ".[dev]"
!uv pip install --system transformers trl jmespath

## 2. Verify Environment

In [ ]:
import sys
sys.path.insert(0, "src")

from sakha.env import SakhaEnvironment
from sakha.models import SakhaAction, ActionType

# Quick smoke test
env = SakhaEnvironment(task="hard")
obs = env.reset()
print(f"Patients: {len(obs.ward_state.patients)}")
print(f"Pending tasks: {obs.pending_count}")
print(f"Step: {obs.ward_state.current_step}")

## 3. Configure Training

Adjust these parameters based on your GPU and time constraints:

In [ ]:
# Training configuration
MODEL = "Qwen/Qwen3-0.6B"          # Model to train (0.6B fits on T4)
TASK = "hard"                      # Task difficulty: easy | medium | hard
EPISODES = 50                        # Number of training episodes
MAX_STEPS = 24                       # Max steps per episode (24 = 2hrs, 96 = full 8hr shift)
SEED = 42                            # Random seed for reproducibility

# GRPO specific
NUM_GENERATIONS = 4                  # Responses per prompt
LEARNING_RATE = 1e-5                 # Learning rate
MAX_COMPLETION_LENGTH = 512          # Max tokens per completion

print(f"Training config:")
print(f"  Model: {MODEL}")
print(f"  Task: {TASK}")
print(f"  Episodes: {EPISODES}")
print(f"  Max steps: {MAX_STEPS}")
print(f"  Estimated time: ~{EPISODES * 2} minutes on T4")

## 4. Define Environment Wrapper for TRL

In [ ]:
from sakha.env import SakhaEnvironment
from sakha.models import SakhaAction, ActionType

class SakhaEnvWrapper:
    """Environment wrapper for TRL's GRPOTrainer."""

    def __init__(self, task: str = "hard", seed: int | None = None):
        self.task = task
        self.seed = seed
        self.env = SakhaEnvironment(task=task)
        self.reward = 0.0
        self._action_type = ActionType
        self._episode_reward = 0.0
        self._episode_steps = 0

    def reset(self, **kwargs) -> str:
        if self.seed is not None:
            import random
            random.seed(self.seed)
        obs = self.env.reset()
        self.reward = 0.0
        self._episode_reward = 0.0
        self._episode_steps = 0
        return self._format_observation(obs)

    def _format_observation(self, obs) -> str:
        pending = obs.ward_state.pending_tasks[:5] if obs.ward_state.pending_tasks else []
        tasks_str = ", ".join(f"{t.required_action}({t.patient_id})" for t in pending) or "No pending tasks"
        return (
            f"Step {obs.ward_state.current_step}, "
            f"Patients: {len(obs.ward_state.patients)}, "
            f"Pending: {obs.pending_count}, "
            f"Tasks: {tasks_str}"
        )

    def _step(self, action) -> str:
        obs = self.env.step(action)
        self.reward = obs.reward or 0.0
        self._episode_reward += self.reward
        self._episode_steps += 1
        return obs

    def review_patient(self, patient_id: int) -> str:
        """Review patient at bed ID. Args: patient_id - bed number."""
        action = SakhaAction(action_type=self._action_type.REVIEW_PATIENT, patient_id=patient_id)
        obs = self._step(action)
        return self._format_observation(obs)

    def administer_medicine(self, patient_id: int) -> str:
        """Administer medication. Args: patient_id - bed number."""
        action = SakhaAction(action_type=self._action_type.ADMINISTER_MEDICINE, patient_id=patient_id)
        obs = self._step(action)
        detail = obs.action_result.detail if obs.action_result else ""
        return f"{self._format_observation(obs)} | {detail}"

    def check_vitals(self, patient_id: int) -> str:
        """Check vitals. Args: patient_id - bed number."""
        action = SakhaAction(action_type=self._action_type.CHECK_VITALS, patient_id=patient_id)
        obs = self._step(action)
        return self._format_observation(obs)

    def alert_doctor(self, patient_id: int) -> str:
        """Alert doctor. Args: patient_id - bed number."""
        action = SakhaAction(action_type=self._action_type.ALERT_DOCTOR, patient_id=patient_id)
        obs = self._step(action)
        return self._format_observation(obs)

    def escalate(self, patient_id: int) -> str:
        """Escalate incident. Args: patient_id - bed number."""
        action = SakhaAction(action_type=self._action_type.ESCALATE, patient_id=patient_id)
        obs = self._step(action)
        return self._format_observation(obs)

    def update_chart(self, patient_id: int) -> str:
        """Update patient chart. Args: patient_id - bed number."""
        action = SakhaAction(action_type=self._action_type.UPDATE_CHART, patient_id=patient_id)
        obs = self._step(action)
        return self._format_observation(obs)

    def prepare_discharge(self, patient_id: int) -> str:
        """Prepare discharge. Args: patient_id - bed number."""
        action = SakhaAction(action_type=self._action_type.PREPARE_DISCHARGE, patient_id=patient_id)
        obs = self._step(action)
        return self._format_observation(obs)

    def medication_round(self) -> str:
        """Complete medication round for all patients."""
        action = SakhaAction(action_type=self._action_type.MEDICATION_ROUND, patient_id=None)
        obs = self._step(action)
        detail = obs.action_result.detail if obs.action_result else ""
        return f"{self._format_observation(obs)} | {detail}"

    def ward_sweep(self) -> str:
        """Complete ward sweep and coordination check."""
        action = SakhaAction(action_type=self._action_type.WARD_SWEEP, patient_id=None)
        obs = self._step(action)
        return self._format_observation(obs)

    def noop(self) -> str:
        """Take no action (pass time)."""
        action = SakhaAction(action_type=self._action_type.NOOP, patient_id=None)
        obs = self._step(action)
        return self._format_observation(obs)

    def get_metrics(self):
        return {
            "episode_reward": self._episode_reward,
            "episode_steps": self._episode_steps,
        }

def reward_func(environments, **kwargs):
    return [getattr(env, "reward", 0.0) for env in environments]

## 5. Create Dataset and Configure GRPO

In [ ]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

def create_prompt(task: str, episode_id: int = 0):
    system_msg = (
        "You are a hospital ward assistant managing patients. "
        "Available actions: review_patient(patient_id), check_vitals(patient_id), "
        "administer_medicine(patient_id), alert_doctor(patient_id), escalate(patient_id), "
        "update_chart(patient_id), prepare_discharge(patient_id), medication_round(), "
        "ward_sweep(), noop(). "
        f"Task difficulty: {task}. "
        f"Episode: {episode_id}. "
        "Choose the best action based on pending tasks and patient needs."
    )
    return [{"role": "system", "content": system_msg}]

# Build dataset
prompts = [create_prompt(TASK, i) for i in range(EPISODES)]
dataset = Dataset.from_dict({"prompt": prompts})

print(f"Dataset size: {len(dataset)} prompts")

In [ ]:
# GRPO configuration
grpo_config = GRPOConfig(
    num_train_epochs=1,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    learning_rate=LEARNING_RATE,
    logging_steps=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    save_strategy="no",
    report_to=[],  # Disable wandb for Colab
    output_dir="./grpo_output",
    seed=SEED,
    remove_unused_columns=False,
)

print("GRPO Config:")
print(f"  Generations: {NUM_GENERATIONS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Max completion: {MAX_COMPLETION_LENGTH}")

## 6. Run Training

In [ ]:
import os
os.environ["TRL_EXPERIMENTAL_SILENCE"] = "1"

trainer = GRPOTrainer(
    model=MODEL,
    train_dataset=dataset,
    reward_funcs=reward_func,
    args=grpo_config,
    environment_factory=lambda: SakhaEnvWrapper(task=TASK, seed=SEED),
)

print(f"Starting training: {EPISODES} episodes, task={TASK}")
print("This will take a while... grab a coffee ☕")

trainer.train()

## 7. Save Results

In [ ]:
import json
from pathlib import Path

results = {
    "model": MODEL,
    "task": TASK,
    "episodes": EPISODES,
    "max_steps": MAX_STEPS,
    "seed": SEED,
    "training_args": {
        "num_generations": NUM_GENERATIONS,
        "learning_rate": LEARNING_RATE,
        "max_completion_length": MAX_COMPLETION_LENGTH,
    },
}

# Save to file
output_path = Path("./grpo_output/results.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(results, indent=2))

print(f"Results saved to {output_path}")
print(json.dumps(results, indent=2))

## 8. Plot Reward Curves

Extract training metrics from the trainer's state log history.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract metrics from trainer logs
history = trainer.state.log_history

steps = []
rewards = []
losses = []
entropies = []

for entry in history:
    if "step" in entry:
        steps.append(entry["step"])
        rewards.append(entry.get("reward", 0))
        losses.append(entry.get("loss", 0))
        entropies.append(entry.get("entropy", 0))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

if steps:
    # Reward plot
    axes[0].plot(steps, rewards, "o-", color="green", linewidth=2)
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Reward")
    axes[0].set_title("Reward over Training")
    axes[0].grid(True, alpha=0.3)

    # Loss plot
    axes[1].plot(steps, losses, "o-", color="red", linewidth=2)
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Loss")
    axes[1].set_title("Training Loss")
    axes[1].grid(True, alpha=0.3)

    # Entropy plot
    axes[2].plot(steps, entropies, "o-", color="blue", linewidth=2)
    axes[2].set_xlabel("Step")
    axes[2].set_ylabel("Entropy")
    axes[2].set_title("Policy Entropy")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("./grpo_output/training_curves.png", dpi=150)
    plt.show()
    print("Plot saved to ./grpo_output/training_curves.png")
else:
    print("No training history found yet. Run training first.")

## 9. Evaluate Trained Model

Run the trained model on a few episodes to see if it learned anything.

In [ ]:
# Quick evaluation
eval_wrapper = SakhaEnvWrapper(task=TASK, seed=SEED + 1000)
obs = eval_wrapper.reset()

print(f"Initial state: {obs}")
print("\nTaking actions...")

for step in range(5):
    # Simple heuristic: review first patient, then noop
    if step == 0:
        result = eval_wrapper.review_patient(1)
    else:
        result = eval_wrapper.noop()
    print(f"Step {step}: reward={eval_wrapper.reward:+.3f} | {result}")

## 10. Download Results

Download the trained model and plots from Colab.

In [ ]:
# Zip results for download
!zip -r grpo_results.zip grpo_output/

from google.colab import files
files.download("grpo_results.zip")

---

## Summary

This notebook demonstrated GRPO training on the Sakha hospital ward environment.

**Key takeaways**:
- Sakha provides a realistic hospital workflow simulation with 96-step episodes
- GRPO enables training LLMs to make tool-use decisions via environment rewards
- Small models (0.6B) can learn basic patterns, but larger models (4B+) show better reasoning

**Next steps**:
- Try larger models (Qwen3-4B) for better performance
- Increase episodes to 200+ for meaningful convergence
- Add eval split to track generalization
- Experiment with different reward shaping